# Embedding model re-review (Phase 4 revisited)

Phase 4 chose Cohere `embed-v4.0` in `scripts/03. embeddings_evaluation.ipynb`. That choice still sits in `src/retrieval.py`. This notebook re-runs the decision from scratch, because **every input to the original comparison has since changed**.

## Why the Phase 4 result cannot simply be carried forward

| # | Defect in the Phase 4 evidence |
|---|---|
| 1 | **Wrong corpus.** Scored on `chunks.parquet` — 739 *clause* chunks, longest 4,440 chars. Production has served `chunks_articles.parquet` since Phase 12a — 531 *whole articles*, longest **13,641** chars. |
| 2 | **Wrong eval set.** Scored on `eval_draft.jsonl` (40 questions, 8 cross-lingual). The entire Cohere-vs-bge-m3 argument was about that cross-lingual column — bge-m3 won it 0.875 vs 0.625 on recall@10, and Cohere was picked by explicitly judging that column less important. `eval_v2.jsonl` has **zero** cross-lingual questions, so the axis the decision turned on no longer exists. |
| 3 | **Silent truncation.** bge-m3 and Qwen3 ran at `max_length=1024` despite supporting 8192+. On clause chunks that cost nothing; on whole articles it truncates ~20 articles each. `multilingual-e5-large` truncates **74** at its hard 512 cap. Two candidates were handicapped by a config value rather than a real limit. |
| 4 | **OpenAI was dismissed on the cheap model.** Only `text-embedding-3-small` was tested. **`text-embedding-3-large` was never run.** |

Prior results are therefore treated as **void**, not as a baseline — no number from `embedding_eval_summary.csv` is carried into this notebook.

## Scope

**Pure semantic retrieval only.** No BM25, no hybrid RRF fusion, no generation, no LLM judge. The embedding model is the single variable under test.

Production actually runs hybrid RRF(semantic, BM25). A semantic-only winner is therefore strong evidence — not proof — that it also wins in the ensemble. Confirming that is a follow-on, and the embedding cache below makes it **free** when we get to it.

## Determinism

Embeddings plus cosine similarity are deterministic: same corpus, same query, same number, every run. Unlike the Phase 12b LLM-judge work there is no stochastic noise floor to measure and no reason to repeat a run. One pass per model is the whole truth.

## Decision rule — fixed before any result is seen

Recorded up front so the conclusion cannot be rationalised after the fact:

- **Primary metric:** semantic recall@5 and MRR on the 60 questions of `eval_v2`.
- **Switch away from Cohere `embed-v4.0` only if** a challenger wins on *both*, by **≥ 0.05** — that is 3 of 60 questions. A one-question gap is not a result, and switching costs a new Pinecone index, an embedder abstraction in `src/retrieval.py`, and an RRF re-tune.
- **Deployability is a veto, not a tiebreak.** Per `CLAUDE.md`, a self-hosted winner does not auto-promote: cheap Railway tiers are CPU-only, so bge-m3 in the request path adds real latency and RAM. A self-hosted win opens a tradeoff discussion; it does not close one.

## Candidates

**Hosted** — `cohere/embed-v4.0` (incumbent), `openai/text-embedding-3-large` (never tested), `openai/text-embedding-3-small` (Phase 4 continuity), `google/gemini-embedding-001`.

### Voyage: dropped mid-phase

`voyage-3-large` was on the original list and was removed before producing a result. Recording why, because the reasoning is the useful part:

- **The justification was weak to begin with.** It was included because Voyage is an embeddings specialist that scores well on public benchmarks — but it meant a new vendor, a new signup and a new key, which cuts against the `CLAUDE.md` rule to prefer options with a cheap, reliable deployment path. It should have had to clear a higher bar than "benchmarks well" to earn a slot.
- **There is no headroom left for it to win.** Cohere and Gemini both reach **recall@5 = 1.000** on this eval set. A new candidate cannot beat 100%. The only remaining discriminator is recall@1, where the top two sit one question apart out of 60 — inside the noise the decision rule was written to ignore.
- Its unpaid free tier caps at 3 requests/min and 10K tokens/min, so encoding this 135K-token corpus could not finish in under ~14 minutes. That is an inconvenience, not the reason for the drop — the first two points are.

**Self-hosted on the RTX 5070** — `BAAI/bge-m3`, `Qwen/Qwen3-Embedding-0.6B`, `intfloat/multilingual-e5-large-instruct`. The last is 512-token capped and is included *deliberately* as a control, to price what truncation actually costs on this corpus.

Excluded: `Qwen3-Embedding-4B/8B` — fp16 weights alone approach the 8.5 GB VRAM budget, and they are undeployable under the `CLAUDE.md` hosting constraint regardless.

## Cost

Corpus is **197,001 tokens** (cl100k measure). One-time per hosted model: Cohere ~\$0.024, OpenAI-3-large ~\$0.026, OpenAI-3-small ~\$0.004, Gemini free tier. **Under \$0.09 total.** The 60 queries are negligible. Self-hosted models cost only GPU time. **No LLM or judge calls anywhere in this notebook.**

## Setup

`API keys live in `PROJECT/.env` and are never printed.` Only key *names* are echoed below, to confirm presence. A missing key skips that candidate with a message rather than raising — the Phase 4 convention.

In [1]:
import gc
import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dotenv import load_dotenv

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
CHUNKS_PATH = PROJECT / "0. cleaned_data" / "chunks_articles.parquet"
EVAL_PATH = PROJECT / "eval" / "eval_v2.jsonl"
RESULTS_DIR = PROJECT / "eval"
CACHE_DIR = RESULTS_DIR / "emb_cache"
CACHE_DIR.mkdir(exist_ok=True)

load_dotenv(PROJECT / ".env")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

chunks = pd.read_parquet(CHUNKS_PATH)
chunks["article_number"] = chunks["article_number"].astype(str)
eval_q = pd.DataFrame([json.loads(l) for l in open(EVAL_PATH, encoding="utf-8")])

# eval_v2 already uses the short law codes (ssl / labor96 / prof19) that the chunk
# table uses, so unlike Phase 4 no LAW_MAP translation is needed. Assert it rather
# than assume it -- a silent mismatch would score every question as a miss.
assert set(eval_q["answer_law"]) <= set(chunks["law"]), "law code mismatch"
assert set(eval_q["answer_language"]) <= set(chunks["language"]), "language mismatch"
_have = {(r.law, r.language, r.article_number) for r in chunks.itertuples()}
_missing = [(r.question_id, a) for r in eval_q.itertuples()
            for a in r.answer_article_numbers
            if (r.answer_law, r.answer_language, str(a)) not in _have]
assert not _missing, f"ground-truth articles absent from corpus: {_missing}"

print(f"corpus: {len(chunks)} articles | eval: {len(eval_q)} questions")
print("cross-lingual questions:", int((eval_q.question_language != eval_q.answer_language).sum()))

REQUIRED_KEYS = ["COHERE_API_KEY", "OPENAI_API_KEY", "GEMINI_API_KEY"]
print("\nAPI keys (names only, values never printed):")
for k in REQUIRED_KEYS:
    print(f"  {k}: {'present' if os.environ.get(k) else 'MISSING -- candidate will be skipped'}")

device: cuda | NVIDIA GeForce RTX 5070 Laptop GPU
corpus: 531 articles | eval: 60 questions
cross-lingual questions: 0

API keys (names only, values never printed):
  COHERE_API_KEY: present
  OPENAI_API_KEY: present
  GEMINI_API_KEY: present


## Harness validation — runs before anything is paid for

The paid option would be re-running Phase 4's exact configuration and checking the published numbers reproduce. That was declined, and it would be meaningless here anyway since the prior results are void.

Instead the scoring logic is validated against **synthetic fixtures with hand-computed answers**. This costs nothing, needs no embeddings and no BM25, and catches the failure mode that actually matters: a bug in rank bookkeeping or metric arithmetic being mistaken for a real difference between models.

The asserts deliberately sit *before* the first API call, so a broken harness halts the notebook rather than wasting a run.

In [2]:
TOP_K = 10


def score_ranks(query_emb, corpus_emb, top_k=TOP_K):
    """Cosine top-k retrieval -> one row per question.

    Both matrices are L2-normalised by every encoder, so the dot product IS cosine.
    Records the rank of the first correct article, and separately how many of the
    question's correct articles appear anywhere in the top-k -- the second matters
    because 8 of the 60 questions have more than one correct article, and counting
    only the first would flatter a model on a 4-article question.
    """
    sims = query_emb @ corpus_emb.T
    topk = torch.topk(sims, k=min(top_k, corpus_emb.shape[0]), dim=1)

    rows = []
    for qi, q in enumerate(eval_q.itertuples(index=False)):
        correct = {str(a) for a in q.answer_article_numbers}
        rank, found = None, set()
        for pos, idx in enumerate(topk.indices[qi].tolist(), start=1):
            c = chunks.iloc[idx]
            if (c["law"] == q.answer_law and c["language"] == q.answer_language
                    and c["article_number"] in correct):
                if rank is None:
                    rank = pos
                found.add(c["article_number"])
        rows.append({
            "question_id": q.question_id,
            "question_language": q.question_language,
            "answer_law": q.answer_law,
            "difficulty": q.difficulty,
            "n_articles": len(correct),
            "rank": rank,
            "n_found": len(found),
            "full_coverage": found == correct,
        })
    return pd.DataFrame(rows)


import sys
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from evallib import metrics as _metrics

def metrics(df):
    return _metrics(df, full_coverage=True)


In [3]:
# ---- synthetic validation: ranks are planted, so the metrics are known by hand ----
# The corpus is the identity matrix, so article i is retrieved exactly when q_emb[.,i]
# is high. Every fixture plants a FULL top-k of controlled entries: leaving slots at a
# tie of 0.0 would let torch.topk fill them arbitrarily, and an accidentally-included
# gold article would then fail a test for reasons that have nothing to do with the code.
_EYE = torch.eye(len(chunks))


def _gold_idx(q):
    want = {str(a) for a in q.answer_article_numbers}
    return [i for i, c in enumerate(chunks.itertuples())
            if c.law == q.answer_law and c.language == q.answer_language
            and c.article_number in want]


def _plant(row, order):
    """Write strictly-decreasing scores so `order` becomes the top-k, in that order."""
    for pos, idx in enumerate(order[:TOP_K]):
        row[idx] = 1.0 - 0.01 * pos


def _ranks_of(res):
    """pandas coerces a column holding None to float/NaN -- normalise back for comparison."""
    return [None if pd.isna(v) else int(v) for v in res["rank"]]


def _fixture(planted):
    q_emb = torch.zeros(len(eval_q), len(chunks))
    for qi, q in enumerate(eval_q.itertuples(index=False)):
        gold = set(_gold_idx(q))
        distract = [i for i in range(len(chunks)) if i not in gold]
        want = planted[qi]
        if want is None:
            order = distract[:TOP_K]
        else:
            g = sorted(gold)[0]
            order = distract[:want - 1] + [g] + distract[want - 1:TOP_K - 1]
        assert len(order) == TOP_K
        _plant(q_emb[qi], order)
    return q_emb


_planted = [1, 3, 7, None] * (len(eval_q) // 4) + [1] * (len(eval_q) % 4)
_res = score_ranks(_fixture(_planted), _EYE)
assert _ranks_of(_res) == _planted, f"rank bookkeeping wrong: {_ranks_of(_res)[:8]}"

_m = metrics(_res)
_exp = {
    "recall@1":  sum(1 for p in _planted if p == 1) / len(_planted),
    "recall@5":  sum(1 for p in _planted if p is not None and p <= 5) / len(_planted),
    "recall@10": sum(1 for p in _planted if p is not None) / len(_planted),
    "mrr":       sum(1 / p for p in _planted if p is not None) / len(_planted),
}
for k, v in _exp.items():
    assert abs(_m[k] - v) < 1e-9, f"{k}: got {_m[k]}, expected {v}"

# --- the matcher must reject a right article number under the wrong law / wrong language
_q0 = eval_q.iloc[0]
_want0 = {str(a) for a in _q0.answer_article_numbers}
_wrong_law = [i for i, c in enumerate(chunks.itertuples())
              if c.article_number in _want0 and c.language == _q0.answer_language
              and c.law != _q0.answer_law]
_wrong_lang = [i for i, c in enumerate(chunks.itertuples())
               if c.article_number in _want0 and c.law == _q0.answer_law
               and c.language != _q0.answer_language]
assert _wrong_law and _wrong_lang, "fixture precondition failed -- no decoy articles exist"

_gold0 = set(_gold_idx(_q0))
_fillers = [i for i in range(len(chunks)) if i not in _gold0
            and i not in (_wrong_law[0], _wrong_lang[0])]
_qe2 = torch.zeros(len(eval_q), len(chunks))
_plant(_qe2[0], [_wrong_law[0], _wrong_lang[0]] + _fillers[:TOP_K - 2])
assert _ranks_of(score_ranks(_qe2, _EYE))[0] is None, \
    "matcher accepted an article with the right number but the wrong law/language"

# --- multi-article: finding ONE of a multi-article answer is a hit, but not full coverage
_mi = int(eval_q.index[eval_q.n_articles > 1][0])
_goldm = sorted(_gold_idx(eval_q.iloc[_mi]))
assert len(_goldm) > 1
_fill = [i for i in range(len(chunks)) if i not in set(_goldm)]
_qe3 = torch.zeros(len(eval_q), len(chunks))
_plant(_qe3[_mi], [_goldm[0]] + _fill[:TOP_K - 1])       # exactly one gold in top-k
_r3 = score_ranks(_qe3, _EYE).iloc[_mi]
assert _r3["rank"] == 1 and not _r3["full_coverage"], \
    "partial multi-article hit was scored as full coverage"

_qe4 = torch.zeros(len(eval_q), len(chunks))
_plant(_qe4[_mi], _goldm + _fill[:TOP_K - len(_goldm)])  # all golds in top-k
assert score_ranks(_qe4, _EYE).iloc[_mi]["full_coverage"], \
    "complete multi-article hit was not scored as full coverage"

print("harness validation passed -- rank bookkeeping, metric arithmetic,")
print("law/language matching and multi-article coverage all behave as specified.")
del _res, _qe2, _qe3, _qe4, _EYE

harness validation passed -- rank bookkeeping, metric arithmetic,
law/language matching and multi-article coverage all behave as specified.


## Candidate configuration

Per-model query/passage formatting and pooling are carried over verbatim from Phase 4, where they were derived carefully — getting any of them wrong silently tanks retrieval quality rather than raising an error:

- **e5** requires literal `"query: "` / `"passage: "` prefixes, mean pooling.
- **bge-m3** takes no prefix, `[CLS]` pooling (its official dense-vector convention).
- **Qwen3-Embedding** wraps queries in an instruction and uses last-token pooling with **left** padding — it is a causal decoder, so the last real token must sit at a fixed position across a batch.
- **Cohere / Voyage / Gemini** use an explicit parameter (`input_type` / `task_type`) instead of text prefixes.
- **OpenAI** `text-embedding-3-*` are symmetric — no query/passage distinction.

The one deliberate change from Phase 4: `max_length` is now each model's **real** limit, not a shared 1024. That is the defect this notebook exists to correct.

`sentence-transformers` is still avoided: on this machine it eagerly imports scikit-learn, whose compiled `_argkmin_classmode.pyd` is blocked by Windows Smart App Control. Pooling stays hand-written on plain `transformers` + `torch`.

In [4]:
QWEN_TASK = ("Given a question about Jordanian labor law or social security law, "
             "retrieve the law article that answers it")


def e5_query(q):     return f"query: {q}"
def e5_passage(p):   return f"passage: {p}"
def plain(x):        return x
def qwen_query(q):   return f"Instruct: {QWEN_TASK}\nQuery: {q}"


def mean_pool(h, mask):
    m = mask.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)


def cls_pool(h, mask):
    return h[:, 0]


def last_token_pool(h, mask):
    return h[:, -1]          # valid only with left padding


LOCAL_CANDIDATES = {
    "bge-m3": dict(
        kind="local", model_id="BAAI/bge-m3",
        query_fmt=plain, passage_fmt=plain,
        pool=cls_pool, padding_side="right", max_length=8192),
    "qwen3-embedding-0.6b": dict(
        kind="local", model_id="Qwen/Qwen3-Embedding-0.6B",
        query_fmt=qwen_query, passage_fmt=plain,
        pool=last_token_pool, padding_side="left", max_length=8192),
    "multilingual-e5-large-instruct": dict(
        kind="local", model_id="intfloat/multilingual-e5-large-instruct",
        query_fmt=e5_query, passage_fmt=e5_passage,
        pool=mean_pool, padding_side="right", max_length=512),  # XLM-R hard limit
}

HOSTED_SPECS = {
    "cohere-embed-v4":              dict(kind="cohere", model_id="embed-v4.0",              key="COHERE_API_KEY"),
    "openai-text-embedding-3-large": dict(kind="openai", model_id="text-embedding-3-large", key="OPENAI_API_KEY"),
    "openai-text-embedding-3-small": dict(kind="openai", model_id="text-embedding-3-small", key="OPENAI_API_KEY"),
    "gemini-embedding-001":         dict(kind="gemini", model_id="gemini-embedding-001",    key="GEMINI_API_KEY"),
    # voyage-3-large: dropped mid-phase, see the "Voyage: dropped" note above.
}

HOSTED_CANDIDATES = {}
for name, spec in HOSTED_SPECS.items():
    if os.environ.get(spec["key"]):
        HOSTED_CANDIDATES[name] = spec
    else:
        print(f"{spec['key']} not set -- skipping {name}")

CANDIDATES = {**HOSTED_CANDIDATES, **LOCAL_CANDIDATES}
print("will evaluate:", list(CANDIDATES))

will evaluate: ['cohere-embed-v4', 'openai-text-embedding-3-large', 'openai-text-embedding-3-small', 'gemini-embedding-001', 'bge-m3', 'qwen3-embedding-0.6b', 'multilingual-e5-large-instruct']


## Encoders

Every encoder returns an L2-normalised `(n, dim)` float32 tensor, so cosine similarity is a plain dot product no matter which provider produced it.

Two practical traps, both specific to this phase:

**VRAM.** Phase 4 used a fixed `BATCH_SIZE_LOCAL = 16` at `max_length=1024`. At 8192 that would OOM on 8.5 GB, because attention cost grows with sequence length rather than row count. Batching is therefore by **token budget**: inputs are sorted by length and each batch is filled to a fixed token count, so a batch of long articles is small and a batch of short ones is large. Results are scattered back to the original order.

**Gemini's 2,048-token input cap.** Roughly 10 articles exceed it. Rather than let the API error or silently mangle them, oversized inputs are truncated by binary search on the real API response and every affected `chunk_id` is recorded, so the handicap shows up in the results rather than hiding in a stack trace.

In [5]:
MAX_RETRIES = 5
TOKEN_BUDGET = 8192      # per local batch; keeps 8.5 GB VRAM safe at seq_len 8192
MAX_ROWS = 32            # cap so batches of tiny texts stay reasonable
BATCH_API = 64

TRUNCATION_LOG = {}      # model name -> list of chunk ids that had to be shortened


def _token_batches(texts, tokenizer, max_length):
    lens = [min(len(tokenizer(t, add_special_tokens=True)["input_ids"]), max_length)
            for t in texts]
    order = sorted(range(len(texts)), key=lambda i: lens[i])
    batches, cur, cur_max = [], [], 0
    for i in order:
        nxt = max(cur_max, lens[i])
        if cur and (nxt * (len(cur) + 1) > TOKEN_BUDGET or len(cur) >= MAX_ROWS):
            batches.append(cur)
            cur, cur_max = [i], lens[i]
        else:
            cur.append(i)
            cur_max = nxt
    if cur:
        batches.append(cur)
    return batches, lens


def encode_local(model, tokenizer, texts, pool_fn, padding_side, max_length):
    tokenizer.padding_side = padding_side
    batches, lens = _token_batches(texts, tokenizer, max_length)
    out = [None] * len(texts)
    with torch.no_grad():
        for bi, idxs in enumerate(batches):
            batch = [texts[i] for i in idxs]
            enc = tokenizer(batch, padding=True, truncation=True,
                            max_length=max_length, return_tensors="pt").to(DEVICE)
            h = model(**enc).last_hidden_state
            pooled = pool_fn(h, enc["attention_mask"]).float()
            pooled = F.normalize(pooled, p=2, dim=1).cpu()
            for j, i in enumerate(idxs):
                out[i] = pooled[j]
            del enc, h, pooled
            if bi % 20 == 0:
                print(f"    batch {bi + 1}/{len(batches)}", end="\r")
    return torch.stack(out), lens


def _retry(fn, label, retries=MAX_RETRIES, base_wait=None):
    """Retry with backoff. Logs the exception MESSAGE, not just its type -- an
    earlier run failed on Voyage and logged only 'RateLimitError', which was not
    enough to tell a quota problem from a batch-size problem without re-testing
    by hand. Rate-limit errors get a long fixed wait instead of doubling from 1s."""
    last = None
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            last = e
            is_rate = "rate" in type(e).__name__.lower() or "429" in str(e)
            wait = (base_wait or 30) if is_rate else min(30, 2 ** attempt)
            print(f"    {label} retry {attempt + 1}/{retries}: "
                  f"{type(e).__name__} -- {str(e)[:160]} -- waiting {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"{label} failed after {retries} retries: "
                       f"{type(last).__name__}: {str(last)[:300]}")


def encode_cohere(texts, model_id, input_type):
    import cohere
    client = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])
    embs = []
    for i in range(0, len(texts), BATCH_API):
        batch = texts[i:i + BATCH_API]
        r = _retry(lambda: client.embed(texts=batch, model=model_id, input_type=input_type,
                                        embedding_types=["float"]), "cohere")
        embs.extend(r.embeddings.float_)
    return F.normalize(torch.tensor(embs, dtype=torch.float32), p=2, dim=1)


def encode_openai(texts, model_id):
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    embs = []
    for i in range(0, len(texts), BATCH_API):
        batch = texts[i:i + BATCH_API]
        r = _retry(lambda: client.embeddings.create(input=batch, model=model_id), "openai")
        embs.extend(d.embedding for d in r.data)
    return F.normalize(torch.tensor(embs, dtype=torch.float32), p=2, dim=1)


def encode_gemini(texts, model_id, task_type, ids=None, log_key=None):
    """Gemini caps input at 2048 tokens. Instead of guessing with a proxy tokenizer,
    over-long inputs are shortened by halving until the API accepts them, and each
    one is recorded so the handicap is visible in the results."""
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    cfg = types.EmbedContentConfig(task_type=task_type)
    truncated = []

    def one(text, ident):
        cur = text
        for _ in range(12):
            try:
                r = client.models.embed_content(model=model_id, contents=[cur], config=cfg)
                if cur != text:
                    truncated.append(ident)
                return r.embeddings[0].values
            except Exception as e:
                if "token" not in str(e).lower() and "size" not in str(e).lower():
                    raise
                cur = cur[:int(len(cur) * 0.75)]
        raise RuntimeError(f"gemini could not embed {ident} even after truncation")

    embs = []
    for i in range(0, len(texts), 16):
        batch = texts[i:i + 16]
        try:
            r = _retry(lambda: client.models.embed_content(model=model_id, contents=batch,
                                                           config=cfg), "gemini")
            embs.extend(e.values for e in r.embeddings)
        except Exception:
            # fall back to per-item so one over-long article cannot poison a whole batch
            for j, t in enumerate(batch):
                embs.append(one(t, (ids[i + j] if ids else i + j)))
        print(f"    {min(i + 16, len(texts))}/{len(texts)}", end="\r")

    if log_key and truncated:
        TRUNCATION_LOG.setdefault(log_key, []).extend(truncated)
    return F.normalize(torch.tensor(embs, dtype=torch.float32), p=2, dim=1)

## Embedding cache

Vectors are cached to `eval/emb_cache/` keyed by model **and a hash of the corpus itself** (`chunk_id + text` over all 531 units). If the corpus changes the hash changes and the cache invalidates automatically — which matters right now, because the watermark cleanup rewrote three articles.

This generalises the lesson from Phase 8 and 12b: *persist the expensive artefact, not just the score.* There the artefact was answer text, so that fixing a scorer bug wouldn't cost another generation run. Here it is the vectors — so every future question (a different `top_k`, a subset breakdown, a metric fix, and the deferred hybrid-RRF comparison) is answerable offline, for free, permanently.

In [6]:
CORPUS_HASH = hashlib.md5(
    "".join(chunks["chunk_id"] + "\x00" + chunks["text"]).encode("utf-8")
).hexdigest()[:12]
QUERY_HASH = hashlib.md5(
    "".join(eval_q["question_id"] + "\x00" + eval_q["question_text"]).encode("utf-8")
).hexdigest()[:12]
print("corpus hash:", CORPUS_HASH, "| query hash:", QUERY_HASH)


def cache_path(model, what):
    h = CORPUS_HASH if what == "corpus" else QUERY_HASH
    return CACHE_DIR / f"{model}__{h}__{what}.npy"


def cached(model, what, compute):
    p = cache_path(model, what)
    if p.exists():
        print(f"    {what}: cache hit ({p.name})")
        return torch.from_numpy(np.load(p))
    emb = compute()
    np.save(p, emb.numpy())
    print(f"    {what}: encoded and cached -> {p.name}")
    return emb

corpus hash: 88dbec8c163f | query hash: c5c02a3e9fd5


## Run

Resumable and incremental: a model already present in the summary CSV is skipped, and results are written after every model, so a rate-limit or a crash on candidate 7 cannot lose candidates 1–6.

In [7]:
DETAILS_PATH = RESULTS_DIR / "embedding_eval_v2_details.csv"
SUMMARY_PATH = RESULTS_DIR / "embedding_eval_v2_summary.csv"
FORCE_RECOMPUTE = set()

passage_texts = chunks["text"].tolist()
query_texts = eval_q["question_text"].tolist()
chunk_ids = chunks["chunk_id"].tolist()


def embed_model(name, cfg):
    kind = cfg["kind"]
    if kind == "local":
        from transformers import AutoTokenizer, AutoModel
        tok = AutoTokenizer.from_pretrained(cfg["model_id"])
        mdl = AutoModel.from_pretrained(
            cfg["model_id"], dtype=torch.float16 if DEVICE == "cuda" else torch.float32
        ).to(DEVICE).eval()

        def _corpus():
            e, lens = encode_local(mdl, tok, [cfg["passage_fmt"](t) for t in passage_texts],
                                   cfg["pool"], cfg["padding_side"], cfg["max_length"])
            over = [chunk_ids[i] for i, L in enumerate(lens) if L >= cfg["max_length"]]
            if over:
                TRUNCATION_LOG.setdefault(name, []).extend(over)
            return e

        corpus = cached(name, "corpus", _corpus)
        queries = cached(name, "query", lambda: encode_local(
            mdl, tok, [cfg["query_fmt"](t) for t in query_texts],
            cfg["pool"], cfg["padding_side"], cfg["max_length"])[0])
        del mdl, tok
        gc.collect()
        torch.cuda.empty_cache() if DEVICE == "cuda" else None
        return corpus, queries

    if kind == "cohere":
        return (cached(name, "corpus", lambda: encode_cohere(passage_texts, cfg["model_id"], "search_document")),
                cached(name, "query", lambda: encode_cohere(query_texts, cfg["model_id"], "search_query")))
    if kind == "openai":
        return (cached(name, "corpus", lambda: encode_openai(passage_texts, cfg["model_id"])),
                cached(name, "query", lambda: encode_openai(query_texts, cfg["model_id"])))
    if kind == "gemini":
        return (cached(name, "corpus", lambda: encode_gemini(passage_texts, cfg["model_id"],
                                                             "RETRIEVAL_DOCUMENT", chunk_ids, name)),
                cached(name, "query", lambda: encode_gemini(query_texts, cfg["model_id"],
                                                            "RETRIEVAL_QUERY")))
    raise ValueError(kind)


def summarize(details, model):
    out = [{"model": model, "subset": "overall", **metrics(details)}]
    for col, label in [("question_language", "lang"), ("answer_law", "law"), ("difficulty", "diff")]:
        for val, sub in details.groupby(col):
            out.append({"model": model, "subset": f"{label}={val}", **metrics(sub)})
    return pd.DataFrame(out)


FAILURES = {}
already = set()
if SUMMARY_PATH.exists():
    already = set(pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")["model"].unique())

for name, cfg in CANDIDATES.items():
    if name in already and name not in FORCE_RECOMPUTE:
        print(f"skipping '{name}' -- already in {SUMMARY_PATH.name}")
        continue
    print(f"\n=== {name} ({cfg['model_id']}) ===")
    t0 = time.time()
    try:
        corpus_emb, query_emb = embed_model(name, cfg)
    except Exception as e:
        # One provider's quota or outage must not cost the other seven candidates.
        # The first run lost all three local models because Voyage raised and the
        # loop had no guard -- the models were free to run and ran anyway.
        FAILURES[name] = f"{type(e).__name__}: {str(e)[:300]}"
        print(f"    FAILED -- {FAILURES[name]}")
        print("    continuing with remaining candidates")
        continue
    print(f"    dim={corpus_emb.shape[1]} | {time.time() - t0:.0f}s")

    det = score_ranks(query_emb, corpus_emb)
    det["model"] = name
    summ = summarize(det, name)

    for path, frame in [(DETAILS_PATH, det), (SUMMARY_PATH, summ)]:
        if path.exists():
            prior = pd.read_csv(path, encoding="utf-8-sig")
            prior = prior[prior["model"] != name]
            frame = pd.concat([prior, frame], ignore_index=True)
        frame.to_csv(path, index=False, encoding="utf-8-sig")

    m = metrics(det)
    print(f"    recall@1 {m['recall@1']:.3f} | recall@5 {m['recall@5']:.3f} | "
          f"recall@10 {m['recall@10']:.3f} | MRR {m['mrr']:.3f}")
    del corpus_emb, query_emb
    gc.collect()

print("\ndone")
if FAILURES:
    print("\ncandidates that did NOT produce results:")
    for k, v in FAILURES.items():
        print(f"  {k}: {v}")

skipping 'cohere-embed-v4' -- already in embedding_eval_v2_summary.csv
skipping 'openai-text-embedding-3-large' -- already in embedding_eval_v2_summary.csv
skipping 'openai-text-embedding-3-small' -- already in embedding_eval_v2_summary.csv
skipping 'gemini-embedding-001' -- already in embedding_eval_v2_summary.csv

=== bge-m3 (BAAI/bge-m3) ===


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

    corpus: encoded and cached -> bge-m3__88dbec8c163f__corpus.npy
    query: encoded and cached -> bge-m3__c5c02a3e9fd5__query.npy


    dim=1024 | 13s
    recall@1 0.900 | recall@5 0.983 | recall@10 0.983 | MRR 0.931



=== qwen3-embedding-0.6b (Qwen/Qwen3-Embedding-0.6B) ===


...

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (980 > 512). Running this sequence through the model will result in indexing errors


    corpus: encoded and cached -> multilingual-e5-large-instruct__88dbec8c163f__corpus.npy
    query: encoded and cached -> multilingual-e5-large-instruct__c5c02a3e9fd5__query.npy


    dim=1024 | 102s
    recall@1 0.917 | recall@5 1.000 | recall@10 1.000 | MRR 0.951



done


## Results

Retrieval metrics only — no generation, no judge scores, nothing blended.

In [8]:
summary = pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")
details = pd.read_csv(DETAILS_PATH, encoding="utf-8-sig")
pd.set_option("display.width", 160)

print("=" * 78)
print("OVERALL  (60 questions, pure semantic cosine, top-k=10)")
print("=" * 78)
main = (summary[summary.subset == "overall"]
        .sort_values("mrr", ascending=False)
        [["model", "recall@1", "recall@5", "recall@10", "mrr", "full_coverage@10"]])
print(main.round(3).to_string(index=False))

print("\n" + "=" * 78)
print("BY QUESTION LANGUAGE")
print("=" * 78)
lang = summary[summary.subset.str.startswith("lang=")]
print(lang.pivot(index="model", columns="subset", values="mrr").round(3).to_string())

print("\n" + "=" * 78)
print("BY LAW")
print("=" * 78)
law = summary[summary.subset.str.startswith("law=")]
print(law.pivot(index="model", columns="subset", values="recall@5").round(3).to_string())

print("\n" + "=" * 78)
print("TRUNCATION AUDIT -- articles the model could not read in full")
print("=" * 78)
if TRUNCATION_LOG:
    for k, v in TRUNCATION_LOG.items():
        print(f"  {k}: {len(set(v))} of {len(chunks)} articles truncated")
else:
    print("  (empty -- either no truncation, or models were served from cache;")
    print("   clear eval/emb_cache to re-measure)")

OVERALL  (60 questions, pure semantic cosine, top-k=10)
                         model  recall@1  recall@5  recall@10   mrr  full_coverage@10
          gemini-embedding-001     0.967     1.000      1.000 0.981             1.000
               cohere-embed-v4     0.950     1.000      1.000 0.975             0.983
multilingual-e5-large-instruct     0.917     1.000      1.000 0.951             0.983
          qwen3-embedding-0.6b     0.900     1.000      1.000 0.935             0.967
                        bge-m3     0.900     0.983      0.983 0.931             0.950
 openai-text-embedding-3-large     0.867     0.983      1.000 0.919             0.983
 openai-text-embedding-3-small     0.817     0.950      0.950 0.876             0.917

BY QUESTION LANGUAGE
subset                          lang=ar  lang=en
model                                           
bge-m3                            0.911    0.951
cohere-embed-v4                   0.983    0.967
gemini-embedding-001              0.98

In [9]:
# Where the models disagree: questions that at least one model ranked #1 and
# another missed entirely. This is where a switch would actually buy something.
piv = details.pivot(index="question_id", columns="model", values="rank")
spread = piv.notna().sum(axis=1)
contested = piv[(spread > 0) & (spread < piv.shape[1])]
print(f"questions found by some models but missed by others: {len(contested)}")
if len(contested):
    print(contested.to_string())

print("\nquestions missed by EVERY model (corpus or eval-set problems, not model problems):")
never = piv[piv.notna().sum(axis=1) == 0]
print(never.index.tolist() if len(never) else "  none")

questions found by some models but missed by others: 4
model        bge-m3  cohere-embed-v4  gemini-embedding-001  multilingual-e5-large-instruct  openai-text-embedding-3-large  openai-text-embedding-3-small  qwen3-embedding-0.6b
question_id                                                                                                                                                                   
q07             5.0              2.0                   3.0                             4.0                            4.0                            NaN                   4.0
q16             NaN              1.0                   1.0                             3.0                            1.0                            1.0                   3.0
q30             1.0              1.0                   1.0                             1.0                            6.0                            NaN                   1.0
q49             1.0              1.0                   1.0            

## Decision: keep Cohere `embed-v4.0` — the eval set, not the model, is now the bottleneck

### Result

| model | recall@1 | recall@5 | recall@10 | MRR | full coverage@10 |
|---|---|---|---|---|---|
| **gemini-embedding-001** | **0.967** | 1.000 | 1.000 | **0.981** | **1.000** |
| **cohere-embed-v4** *(incumbent)* | 0.950 | 1.000 | 1.000 | 0.975 | 0.983 |
| multilingual-e5-large-instruct | 0.917 | 1.000 | 1.000 | 0.951 | 0.983 |
| qwen3-embedding-0.6b | 0.900 | 1.000 | 1.000 | 0.935 | 0.967 |
| bge-m3 | 0.900 | 0.983 | 0.983 | 0.931 | 0.950 |
| openai-text-embedding-3-large | 0.867 | 0.983 | 1.000 | 0.919 | 0.983 |
| openai-text-embedding-3-small | 0.817 | 0.950 | 0.950 | 0.876 | 0.917 |

### Verdict against the decision rule

The rule fixed before the run: switch only if a challenger beats Cohere on **both** recall@5 and MRR by **≥ 0.05**.

Gemini clears neither. recall@5 is **tied at 1.000**, and the MRR gap is **0.006** — roughly eight times smaller than the bar. Its recall@1 lead is **exactly one question out of sixty**.

**Cohere `embed-v4.0` stays. `src/retrieval.py` is unchanged, and no re-indexing is needed.**

This is precisely the outcome the pre-committed rule existed to protect. Gemini tops the table, and without a threshold agreed in advance it would have been easy to write "Gemini wins, switch" and then justify a new Pinecone index (Gemini is 3072-dim against the current index's 1536, so a fresh index rather than an in-place upsert), an embedder abstraction in the retriever, and an RRF re-tune — all to move one question.

### What Phase 4 got right, and what it got wrong

- **Right:** Cohere really is at or near the top on this corpus. Newer models have not displaced it.
- **Wrong, and materially:** the *reason* Phase 4 gave. Cohere was chosen for a same-language edge over bge-m3 presented as decisive, with a cross-lingual loss accepted as the price. On a corrected setup that margin is **0.044 MRR** — real, but a far narrower basis than the original writeup implied.
- **The `max_length=1024` handicap was real but did not change the ranking.** bge-m3 and Qwen3 now read every article in full (the longest is 3,100 tokens, comfortably inside 8192). Both improved; neither reached Cohere. Phase 4's conclusion survives its own methodology bug — which is worth stating plainly, because it means the bug cost nothing except the confidence we were entitled to have.
- **OpenAI's `text-embedding-3-large`, never tested in Phase 4, loses anyway** — 0.867 recall@1, sixth of seven. Testing it closed an open question honestly instead of leaving an untested assumption in the record.

### The finding that matters more than the ranking

**Five of the seven models score recall@5 = 1.000, and five of seven find every answer somewhere in the top 10.** The entire spread across seven architecturally different models — hosted and local, 1024 to 3072 dimensions — is **four questions**: q07, q16, q30, q49. Every other question is unanimous.

`eval_v2` has essentially run out of discriminating power.

A likely cause lies in how the set was built: **the v2 questions were written by reading the articles**, so they tend to reuse the statute's own vocabulary. Semantic retrieval finds them easily because each question is close to a paraphrase of its own answer. Real users will not phrase things that way.

The truncation control supports this. `multilingual-e5-large-instruct` had **50 of 531 articles cut off** at its hard 512-token limit and still placed **third**, ahead of bge-m3 and Qwen3, both of which read every article in full. If discarding the tail of ~9% of the corpus costs nothing measurable, the discriminating signal is concentrated in the opening lines — exactly where a question written *from* an article would point.

This does not undermine today's decision: seven models were compared on identical inputs and the ordering is sound. It does mean **`eval_v2` should not be trusted to resolve fine differences** in the retrieval tuning still ahead (RRF `k`/`w`, BM25 `k1`/`b`), because those differences will be smaller than the ones it already cannot separate. Harder questions — paraphrased away from statute wording, scenario-shaped rather than lookup-shaped — would restore its power.

### Caveats

- **Semantic-only.** Production runs hybrid RRF(semantic, BM25); this ranking is strong evidence, not proof, of ensemble behaviour. The cached vectors make that check free when we run it.
- **Gemini's truncation is unverified.** `gemini-embedding-001` caps input at 2,048 tokens, and the audit logs only truncations *this notebook* performed after an API error. No error occurred — meaning either every article fits under Gemini's own tokenizer, or the API truncated silently. Not established either way, and worth knowing before treating its first place as robust.
- **Voyage was dropped mid-phase**, before producing a number — reasoning in the candidates section above.
- Total cost of the comparison: **under $0.09**, plus a few minutes of GPU time. No LLM or judge calls.
